# OpenWebUI Trace DEA Optimization Experiments

This notebook drives bounded experiments for `trace_openwebui_dea_skeleton.py`. It is intentionally parameterized so expensive OpenWebUI/DEA runs are opt-in.

Set `RUN_OPENWEBUI_TRACE_EXPERIMENTS=1` to execute commands. Without it, the notebook validates and displays the experiment plan only.

## Objective Evidence Matrix

| Requirement | Current evidence | Runtime caveat |
|---|---|---|
| Optimize model parameters | `OpenWebUIModel`, model trainer test, notebook `offline_model_step3_calibration` | Live quality benchmark still depends on model availability and inference speed |
| Optimize tool parameters | `OpenWebUISummarizerAgent`, tool forwarding tests, notebook `offline_tool_step3_calibration` | Actual effect depends on the OpenWebUI pipe/tool consuming the forwarded argument or valve |
| Step 2 path | Trainer test covers bridge generation followed by offline PromptFoo on `step2_candidate_rows.csv` | Full live generation still depends on OpenWebUI/backend health |
| Step 3 path | Trainer test covers prepared live rows and PromptFoo scoring artifacts | PromptFoo `json.output` transforms can hide bridge internals unless trace sidecars are added |
| Direct DEA with DEA judge | DEA guide test verifies `use_dea_judge=True` forwarding and trace metrics aggregation | Real judge latency/cost depends on the configured judge model/provider |
| Batch-of-3 optimization | Tests and notebook artifacts use `--batch-size 3`; summaries record `batch_size` | Larger profiles scale linearly with model/tool runtime |
| Trace and feedback | Step artifacts include `.summary.json`, `.traces.json`, candidates CSV, feedback text, scores, params, durations | Step 3 live PromptFoo records command/runtime unless bridge trace is separately persisted |
| Faster algorithm-specific tools | Notebook and tests cover `summarizer---dtcrs`, `summarizer---kohaku`, `summarizer---kohaku-OR`, `summarizer---lightrag`, `summarizer---raptor` | OpenWebUI permissions/upstream quota must allow each pipe to execute |


In [1]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts/promptfoo_openwebui_eval/trace_openwebui_dea_skeleton.py').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

REPO_ROOT = find_repo_root(Path.cwd())
BUNDLE = REPO_ROOT / 'scripts/promptfoo_openwebui_eval'
SCRIPT = BUNDLE / 'trace_openwebui_dea_skeleton.py'
INPUT_CSV = BUNDLE / 'datasets/bigsurvey/step2_input.csv'
RUN_LIVE = os.environ.get('RUN_OPENWEBUI_TRACE_EXPERIMENTS') == '1'
PROFILE = os.environ.get('TRACE_EXPERIMENT_PROFILE', 'smoke')
RUN_ID = os.environ.get('TRACE_EXPERIMENT_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = BUNDLE / 'results' / 'trace_experiments' / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def show_table(rows: list[dict[str, Any]], title: str = '') -> None:
    if title:
        display(Markdown(f'### {title}'))
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        display(Markdown('```json\n' + json.dumps(rows, indent=2, ensure_ascii=False) + '\n```'))

print({'repo_root': str(REPO_ROOT), 'bundle': str(BUNDLE), 'run_live': RUN_LIVE, 'profile': PROFILE, 'results_dir': str(RESULTS_DIR)})

{'repo_root': '/home/xav/code/document_embedding_analysis', 'bundle': '/home/xav/code/document_embedding_analysis/scripts/promptfoo_openwebui_eval', 'run_live': True, 'profile': 'offline_calibration', 'results_dir': '/home/xav/code/document_embedding_analysis/scripts/promptfoo_openwebui_eval/results/trace_experiments/20260613_142220'}


In [2]:
tool_models = [
    {'algorithm': 'dtcrs', 'target_model_id': 'summarizer---dtcrs'},
    {'algorithm': 'kohaku', 'target_model_id': 'summarizer---kohaku'},
    {'algorithm': 'kohaku_openrouter', 'target_model_id': 'summarizer---kohaku-OR'},
    {'algorithm': 'lightrag', 'target_model_id': 'summarizer---lightrag'},
    {'algorithm': 'raptor', 'target_model_id': 'summarizer---raptor'},
]

method_matrix = [
    {'method': 'direct_dea', 'generation': 'bridge', 'evaluation': 'native DEA', 'trace': 'bridge trace + total runtime', 'cost': 'medium'},
    {'method': 'direct_dea_judge', 'generation': 'bridge', 'evaluation': 'native DEA + DEA LLM judge', 'trace': 'bridge trace + judge runtime', 'cost': 'high'},
    {'method': 'direct_llm_judge', 'generation': 'bridge', 'evaluation': 'custom JSON LLM judge', 'trace': 'bridge trace + judge runtime', 'cost': 'medium/high'},
    {'method': 'step2_promptfoo', 'generation': 'bridge first', 'evaluation': 'PromptFoo offline CSV', 'trace': 'bridge trace + PromptFoo artifacts', 'cost': 'high'},
    {'method': 'step3_promptfoo', 'generation': 'PromptFoo live bridge provider', 'evaluation': 'PromptFoo live', 'trace': 'PromptFoo artifacts + command runtime', 'cost': 'high'},
]

show_table(tool_models, 'Tool model ids')
show_table(method_matrix, 'Optimization methods')

### Tool model ids

,algorithm,target_model_id
0,dtcrs,summarizer---dtcrs
1,kohaku,summarizer---kohaku
2,kohaku_openrouter,summarizer---kohaku-OR
3,lightrag,summarizer---lightrag
4,raptor,summarizer---raptor


### Optimization methods

,method,generation,evaluation,trace,cost
0,direct_dea,bridge,native DEA,bridge trace + total runtime,medium
1,direct_dea_judge,bridge,native DEA + DEA LLM judge,bridge trace + judge runtime,high
2,direct_llm_judge,bridge,custom JSON LLM judge,bridge trace + judge runtime,medium/high
3,step2_promptfoo,bridge first,PromptFoo offline CSV,bridge trace + PromptFoo artifacts,high
4,step3_promptfoo,PromptFoo live bridge provider,PromptFoo live,PromptFoo artifacts + command runtime,high


In [3]:
def base_command(exp: dict[str, Any]) -> list[str]:
    output_json = RESULTS_DIR / f"{exp['name']}.json"
    artifact_dir = RESULTS_DIR / exp['name']
    args = [
        'python', str(SCRIPT),
        '--input-csv', str(INPUT_CSV),
        '--max-rows', str(exp.get('max_rows', 3)),
        '--batch-size', str(exp.get('batch_size', 3)),
        '--num-epochs', str(exp.get('num_epochs', 1)),
        '--optimization-method', exp['method'],
        '--optimize-target', exp['optimize_target'],
        '--target-model-id', exp['target_model_id'],
        '--bridge-timeout-seconds', str(exp.get('bridge_timeout_seconds', 600)),
        '--artifact-dir', str(artifact_dir),
        '--output-json', str(output_json),
        '--optimizer-mode', exp.get('optimizer_mode', 'noop'),
        '--optimizer-max-tokens', str(exp.get('optimizer_max_tokens', 512)),
    ]
    if exp['optimize_target'] == 'tool':
        args += ['--initial-tool-params-json', json.dumps(exp.get('tool_params', {}), ensure_ascii=False)]
    else:
        args += ['--initial-system-prompt', exp.get('system_prompt', 'Answer with concise, evidence-grounded synthesis.'), '--initial-model-params-json', json.dumps(exp.get('model_params', {}))]
    if exp['method'] in {'step2_promptfoo', 'step3_promptfoo'}:
        step = 'step3' if exp['method'] == 'step3_promptfoo' else 'step2'
        args += [
            '--promptfoo-config', f'bigsurvey.{step}.dea.yaml',
            '--promptfoo-workdir', str(BUNDLE),
            '--promptfoo-eval-cmd', exp.get('promptfoo_eval_cmd', 'promptfoo eval -c {config} -t {csv} --output {result_json} --no-progress-bar'),
        ]
    if exp['method'] == 'direct_llm_judge':
        args += [
            '--judge-base-url', exp.get('judge_base_url', 'http://127.0.0.1:8001/v1'),
            '--judge-api-key', exp.get('judge_api_key', 'dummy'),
            '--judge-model', exp.get('judge_model', 'qwen-laptop-dea:latest'),
            '--judge-timeout-seconds', str(exp.get('judge_timeout_seconds', 180)),
            '--judge-extra-body-json', json.dumps(exp.get('judge_extra_body', {'max_tokens': 192}), ensure_ascii=False),
            '--judge-system-prompt', exp.get('judge_system_prompt', 'Return compact strict JSON only.'),
            '--judge-prompt-template', exp.get('judge_prompt_template', 'Task:\n{task_text}\nReference:\n{reference_text}\nCandidate:\n{candidate_answer}\nReturn JSON only: {{"score": 0.0, "feedback": "...", "metrics": {{"correctness": 0.0}}}}'),
        ]
    return args

def experiment_rows(profile: str) -> list[dict[str, Any]]:
    kohaku = {'algorithm': 'kohaku', 'target_model_id': 'summarizer---kohaku'}
    smoke = [
        {'name': 'smoke_model_local_direct_llm_judge', 'method': 'direct_llm_judge', 'optimize_target': 'model', 'target_model_id': 'qwen-laptop:latest', 'optimizer_mode': 'noop', 'model_params': {'generation_temperature': 0, 'generation_max_tokens': 64}, 'system_prompt': 'Answer concisely and include only the requested answer.'}
    ]
    if profile == 'smoke':
        return smoke
    fake_promptfoo_cmd = "python -c \"import json,pathlib,sys; pathlib.Path(sys.argv[1]).write_text(json.dumps({{'score': 0.73, 'namedScores': {{'promptfoo_score': 0.73}}}}), encoding='utf-8')\" {result_json}"
    if profile == 'offline_calibration':
        return [
            {'name': 'offline_model_step3_calibration', 'method': 'step3_promptfoo', 'optimize_target': 'model', 'target_model_id': 'qwen-laptop:latest', 'optimizer_mode': 'noop', 'model_params': {'generation_temperature': 0, 'generation_max_tokens': 64}, 'system_prompt': 'Calibration run only.', 'promptfoo_eval_cmd': fake_promptfoo_cmd},
            {'name': 'offline_tool_step3_calibration', 'method': 'step3_promptfoo', 'optimize_target': 'tool', 'target_model_id': 'summarizer---kohaku', 'optimizer_mode': 'noop', 'tool_params': {'algorithm': 'kohaku', 'target_length': 'short', 'structure': 'thematic'}, 'promptfoo_eval_cmd': fake_promptfoo_cmd},
        ]
    if profile == 'method_check':
        return [
            {'name': f"method_{m['method']}_kohaku", 'method': m['method'], 'optimize_target': 'tool', 'target_model_id': kohaku['target_model_id'], 'tool_params': {'algorithm': 'kohaku', 'target_length': 'long', 'structure': 'thematic'}}
            for m in method_matrix
        ]
    if profile == 'algorithm_grid':
        return [
            {'name': f"alg_{m['algorithm']}_direct_dea_judge", 'method': 'direct_dea_judge', 'optimize_target': 'tool', 'target_model_id': m['target_model_id'], 'tool_params': {'algorithm': m['algorithm'].replace('_openrouter', ''), 'target_length': 'long', 'structure': 'thematic'}}
            for m in tool_models
        ]
    if profile == 'model_smoke':
        return [
            {'name': 'model_qwen3627b_direct_dea', 'method': 'direct_dea', 'optimize_target': 'model', 'target_model_id': 'qwen3627b', 'model_params': {'generation_temperature': 0.2, 'generation_top_p': 0.95, 'generation_max_tokens': 4096}}
        ]
    return smoke

experiments = experiment_rows(PROFILE)
plan_rows = []
for exp in experiments:
    cmd = base_command(exp)
    plan_rows.append({**exp, 'command': ' '.join(shlex.quote(x) for x in cmd)})

show_table(plan_rows, f'Experiment plan: {PROFILE}')
if pd is not None:
    pd.DataFrame(plan_rows).to_csv(RESULTS_DIR / 'experiment_plan.csv', index=False)

### Experiment plan: offline_calibration

,name,method,optimize_target,target_model_id,optimizer_mode,model_params,system_prompt,promptfoo_eval_cmd,command,tool_params
0,offline_model_step3_calibration,step3_promptfoo,model,qwen-laptop:latest,noop,"{'generation_temperature': 0, 'generation_max_...",Calibration run only.,"python -c ""import json,pathlib,sys; pathlib.Pa...",python /home/xav/code/document_embedding_analy...,NaN
1,offline_tool_step3_calibration,step3_promptfoo,tool,summarizer---kohaku,noop,NaN,NaN,"python -c ""import json,pathlib,sys; pathlib.Pa...",python /home/xav/code/document_embedding_analy...,"{'algorithm': 'kohaku', 'target_length': 'shor..."


In [4]:
def summarize_output(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding='utf-8'))
    best = payload.get('best_weighted') or {}
    scores = best.get('mean_scores') or {}
    state = best.get('current_state') or {}
    return {
        'best_scalar_objective': best.get('scalar_objective'),
        'score': scores.get('score'),
        'dea_composite': scores.get('dea_composite'),
        'plan': scores.get('plan'),
        'content': scores.get('content'),
        'resources': scores.get('resources'),
        'length_alignment': scores.get('length_alignment'),
        'execution_duration_s': scores.get('execution_duration_s'),
        'promptfoo_duration_s': scores.get('promptfoo_duration_s'),
        'llm_calls': scores.get('llm_calls'),
        'tool_calls': scores.get('tool_calls'),
        'batch_size': best.get('batch_size'),
        'optimizer_mode': best.get('optimizer_mode'),
        'param_json': state.get('param_json'),
        'model_param_json': state.get('model_param_json'),
        'tool_param_json': state.get('tool_param_json'),
        'system_prompt_chars': len(state.get('system_prompt') or ''),
    }

results = []
for exp in experiments:
    cmd = base_command(exp)
    output_json = RESULTS_DIR / f"{exp['name']}.json"
    row = {'name': exp['name'], 'method': exp['method'], 'target_model_id': exp['target_model_id'], 'status': 'planned'}
    if RUN_LIVE:
        t0 = time.time()
        env = dict(os.environ, DEA_REPO_ROOT=str(REPO_ROOT), DEA_REPO_PATH=str(REPO_ROOT))
        proc = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env, text=True, capture_output=True, timeout=int(os.environ.get('TRACE_EXPERIMENT_TIMEOUT_SECONDS', '7200')))
        stdout_path = RESULTS_DIR / f"{exp['name']}.stdout.txt"
        stderr_path = RESULTS_DIR / f"{exp['name']}.stderr.txt"
        stdout_path.write_text(proc.stdout, encoding='utf-8')
        stderr_path.write_text(proc.stderr, encoding='utf-8')
        row.update({'status': 'ok' if proc.returncode == 0 else 'failed', 'returncode': proc.returncode, 'wall_s': time.time() - t0, 'log_stdout': str(stdout_path), 'log_stderr': str(stderr_path), 'error_tail': proc.stderr[-1000:] if proc.returncode else ''})
        row.update(summarize_output(output_json))
    results.append(row)
    show_table([row], f"Result: {exp['name']}")

show_table(results, 'Summary results')
if pd is not None:
    pd.DataFrame(results).to_csv(RESULTS_DIR / 'experiment_results.csv', index=False)
else:
    (RESULTS_DIR / 'experiment_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

### Result: offline_model_step3_calibration

,name,method,target_model_id,status,returncode,wall_s,log_stdout,log_stderr,error_tail,best_scalar_objective,...,execution_duration_s,promptfoo_duration_s,llm_calls,tool_calls,batch_size,optimizer_mode,param_json,model_param_json,tool_param_json,system_prompt_chars
0,offline_model_step3_calibration,step3_promptfoo,qwen-laptop:latest,ok,0,0.64191,/home/xav/code/document_embedding_analysis/scr...,/home/xav/code/document_embedding_analysis/scr...,,0.95569,...,0.029582,0.021988,0.0,0.0,3,noop,"{""generation_temperature"": 0.0, ""generation_ma...","{""generation_temperature"": 0.0, ""generation_ma...",,21


### Result: offline_tool_step3_calibration

,name,method,target_model_id,status,returncode,wall_s,log_stdout,log_stderr,error_tail,best_scalar_objective,...,execution_duration_s,promptfoo_duration_s,llm_calls,tool_calls,batch_size,optimizer_mode,param_json,model_param_json,tool_param_json,system_prompt_chars
0,offline_tool_step3_calibration,step3_promptfoo,summarizer---kohaku,ok,0,0.50102,/home/xav/code/document_embedding_analysis/scr...,/home/xav/code/document_embedding_analysis/scr...,,0.956088,...,0.026778,0.020008,0.0,0.0,3,noop,"{""algorithm"": ""kohaku"", ""target_length"": ""shor...",,"{""algorithm"": ""kohaku"", ""target_length"": ""shor...",0


### Summary results

,name,method,target_model_id,status,returncode,wall_s,log_stdout,log_stderr,error_tail,best_scalar_objective,...,execution_duration_s,promptfoo_duration_s,llm_calls,tool_calls,batch_size,optimizer_mode,param_json,model_param_json,tool_param_json,system_prompt_chars
0,offline_model_step3_calibration,step3_promptfoo,qwen-laptop:latest,ok,0,0.64191,/home/xav/code/document_embedding_analysis/scr...,/home/xav/code/document_embedding_analysis/scr...,,0.955690,...,0.029582,0.021988,0.0,0.0,3,noop,"{""generation_temperature"": 0.0, ""generation_ma...","{""generation_temperature"": 0.0, ""generation_ma...",,21
1,offline_tool_step3_calibration,step3_promptfoo,summarizer---kohaku,ok,0,0.50102,/home/xav/code/document_embedding_analysis/scr...,/home/xav/code/document_embedding_analysis/scr...,,0.956088,...,0.026778,0.020008,0.0,0.0,3,noop,"{""algorithm"": ""kohaku"", ""target_length"": ""shor...",,"{""algorithm"": ""kohaku"", ""target_length"": ""shor...",0
